# reglscatterpy — feature tour

Interactive WebGL scatterplots for single-cell data, with a **scanpy-style** API.

Plots render **static by default** (a self-contained snapshot that reopens with no kernel, like a plotly figure). Pass **`interactive=True`** for the live, kernel-linked widget that round-trips a lasso selection back to Python (`w.selection`).

Interactions: drag to pan, **Ctrl/Cmd + scroll to zoom** (plain scroll scrolls the page), **double-click to reset** the view.

## Setup — a synthetic single-cell dataset
Clusters with marker genes, a log-normalised `.raw` and a scaled `.X` (so the `use_raw` demo is meaningful), plus UMAP/PCA embeddings and numeric QC columns.

In [ ]:
import numpy as np, pandas as pd, anndata as ad
import reglscatterpy as rs

rng = np.random.default_rng(0)
n, g, k = 3000, 50, 6
labels = rng.integers(0, k, n)
centers = rng.normal(0, 6, (k, 2))
umap = centers[labels] + rng.normal(0, 1.2, (n, 2))
counts = rng.poisson(0.5, (n, g)).astype(float)
for c in range(k):
    m = slice(c*6, c*6+3)            # 3 marker genes per cluster
    counts[labels == c, m] += rng.poisson(10, (int((labels == c).sum()), 3))
lib = counts.sum(1, keepdims=True); lib[lib == 0] = 1
lognorm = np.log1p(counts / lib * 1e4)             # .raw  (0..~5)
X = (lognorm - lognorm.mean(0)) / (lognorm.std(0) + 1e-9)   # .X (z-scored)
var = pd.DataFrame(index=[f'Gene{i}' for i in range(g)])
obs = pd.DataFrame({
    'cluster': pd.Categorical([f'C{l}' for l in labels]),
    'n_counts': counts.sum(1), 'n_genes': (counts > 0).sum(1),
    'pct_mito': rng.beta(1.5, 20, n) * 100, 'score': rng.random(n),
}, index=[f'cell{i}' for i in range(n)])
adata = ad.AnnData(X=X, obs=obs, var=var)
adata.raw = ad.AnnData(X=lognorm, var=var)
adata.obsm['X_umap'] = umap
adata.obsm['X_pca'] = rng.normal(0, 1, (n, 10))
adata

## 1. Basics
`basis=` picks the embedding (short names like `'umap'`/`'pca'` resolve to the `obsm` key). `color=` takes an `obs` column or a gene.

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster')   # an obs column

In [ ]:
rs.scatterplot(adata, basis='umap', color='Gene0')     # a gene

## 2. Colour scales
`cmap` (continuous), `palette` (categorical), `vmin`/`vmax` (numbers or `'p1'`/`'p99'` percentiles), `center_zero`.

In [ ]:
rs.scatterplot(adata, basis='umap', color='Gene0', cmap='magma', vmax='p99')

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster', palette='Dark2')

### `use_raw` — matching scanpy's colour scale
Like `sc.pl.umap`, a gene defaults to `.raw` (log-normalised, e.g. 0–5). Pass `use_raw=False` to colour by the scaled `.X` (z-scored, negative). Watch the colour bar.

In [ ]:
rs.scatterplot(adata, basis='umap', color='Gene0')                 # .raw (default)

In [ ]:
rs.scatterplot(adata, basis='umap', color='Gene0', use_raw=False)  # scaled .X

## 3. Size & opacity
`size=` is a scalar **or** a column/gene name (per-point). `opacity_by=` likewise.

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster', size=8)

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster', size='n_counts')   # size by a column

## 4. Draw order (z-depth)
`sort_order=True` (default) draws higher continuous values on top. `random_state=N` shuffles the draw order (seeded) so no category is systematically hidden by overplotting.

In [ ]:
rs.scatterplot(adata, basis='umap', color='Gene0', sort_order=True)   # high values on top

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster', random_state=0)  # seeded shuffle

## 5. `na_color` & `groups`
`groups=[...]` keeps only those categories coloured; the rest grey out to `na_color`.

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster', groups=['C0', 'C3'], na_color='#dddddd')

## 6. Other components / embeddings
`components=(i, j)` is 1-based (scanpy). Plot PC2 vs PC3:

In [ ]:
rs.scatterplot(adata, basis='pca', color='cluster', components=(2, 3))

## Big data — atlas-scale rendering
`scatterplot()` stays interactive on huge data **without silently hiding cells**. Let's make a larger synthetic atlas to demo the three modes.

In [ ]:
bn, bk = 1_500_000, 12
blab = rng.integers(0, bk, bn)
bcent = rng.normal(0, 8, (bk, 2))
bumap = bcent[blab] + rng.normal(0, 1.0, (bn, 2))
big = ad.AnnData(
    X=np.zeros((bn, 1), dtype='float32'),
    obs=pd.DataFrame({'cell_type': pd.Categorical([f'T{l}' for l in blab]),
                      'score': rng.random(bn).astype('float32')}),
)
big.obsm['X_umap'] = bumap.astype('float32')
big

### Auto density subsample (default)
`max_points='auto'` (the default) caps at 500k via a **density-preserving** sketch that thins dense blobs but keeps rare types. It's honest about it: an on-plot `'X of Y shown'` caption + a one-time warning. `subsample='random'` is the uniform fallback.

In [ ]:
rs.scatterplot(big, basis='umap', color='cell_type')   # caption + downsample warning

### All points resident (ABC-Atlas style)
`max_points=None` puts **every** cell on the GPU; pan/zoom is camera-only. Smooth up to ~4M points on a decent GPU.

In [ ]:
rs.scatterplot(big, basis='umap', color='cell_type', max_points=None)

### Progressive detail-on-zoom (>4M)
`progressive=True` shows a light density overview, then re-renders **all cells inside the viewport** as you zoom in (no preprocessing; always the live widget). Tune with `progressive_opts={'detail_max_points': ..., 'overscan': ...}` (points per viewport / margin fetched).

In [ ]:
rs.scatterplot(big, basis='umap', color='cell_type', progressive=True,
               progressive_opts={'detail_max_points': 300_000, 'overscan': 0.6})

## 7. Multi-panel grid
A **list** of names → one linked panel per value (genes and/or obs), camera + lasso synced. `ncols` sets the columns. (A single-element list is just one plot.)

In [ ]:
rs.scatterplot(adata, basis='umap', color=['cluster', 'Gene0', 'Gene6'], ncols=3)

## 8. Filtering
`filter_by=` shows distribution sliders: drag the **black grabbers**, or drag the **band** to pan the range; filtering is **live** while you drag (auto-deferred above 150k points).

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster',
               filter_by=adata.obs[['n_counts', 'n_genes', 'pct_mito']])

## 9. Richer tooltips
`tooltip_by=` adds fields on hover (obs columns or genes).

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster',
               tooltip_by=['n_genes', 'pct_mito', 'Gene0'])

## 10. Toolbar & view
`toolbar='left'`/`'top'`/`'none'`: pan, lasso, zoom-to-selection, reset, screenshot. `zoom_on_selection=True` auto-frames a lasso. Double-click resets; Ctrl/Cmd+scroll zooms.

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster', toolbar='left', zoom_on_selection=True)

## 11. Interactive selection round-trip 🔴 *(needs `interactive=True` + a live kernel)*
This is the live widget. **Lasso** some cells in the plot (toolbar → lasso), then read them back in the next cell. You can also set the selection from Python.

In [ ]:
w = rs.scatterplot(adata, basis='umap', color='cluster', interactive=True, toolbar='left')
w   # lasso a cluster, then run the cells below

In [ ]:
# the lassoed cells, as positional indices into adata (empty until you lasso):
w.selection

In [ ]:
# drive it from Python too — this highlights those points in the plot above:
w.selection = list(np.where(adata.obs['cluster'] == 'C0')[0])
len(w.selection)

### Analyse the selection
`subset`, `diff_expression` (uses scanpy `rank_genes_groups` when installed, else a Welch t-test), `composition`, and `annotate` (writes back to `adata.obs`).

In [ ]:
sub = w.subset()          # adata[w.selection]
sub

In [ ]:
w.diff_expression(n=5)    # top markers of the selection vs the rest

In [ ]:
w.composition('cluster')  # what clusters the selection is made of

In [ ]:
w.annotate('my_label', 'group A')   # writes adata.obs['my_label'] for the selected cells
rs.scatterplot(adata, basis='umap', color='my_label')

## 12. Linked grid (`compose`)
Pass plots to `rs.compose(...)` — pan/zoom and lasso stay in sync across panels. `compose` makes the panels interactive for you (no need for `interactive=True` on each).

In [ ]:
a = rs.scatterplot(adata, basis='umap', color='cluster')
b = rs.scatterplot(adata, basis='pca',  color='cluster')
rs.compose([a, b])

## 13. Width
Plots are **700 px** by default. Pass `width=` (px), or `width=None` to fill the cell.

In [ ]:
rs.scatterplot(adata, basis='umap', color='cluster', width=400)

## 14. Other inputs & export
DataFrames (give `x`/`y` columns) and numpy arrays work too. `save='plot.html'` writes a self-contained offline file; `w.to_html(...)` does the same for one plot.

In [ ]:
df = pd.DataFrame({'x': umap[:, 0], 'y': umap[:, 1], 'cluster': obs['cluster'].values})
rs.scatterplot(df, x='x', y='y', color='cluster')

In [ ]:
import tempfile, os
out = os.path.join(tempfile.gettempdir(), 'umap.html')
rs.scatterplot(adata, basis='umap', color='cluster', save=out)
print('wrote', out)

---
**Static vs interactive:** the default static plots reopen with no kernel (great for sharing / reports); `interactive=True` adds the Python round-trip but, like any Jupyter widget, needs a running kernel to render. Use the default for figures you keep, `interactive=True` while actively selecting.